**Выгрузка численности населения с сайта Росстата Иркутской области**

- Автор: Клинкова Екатерина
- Telegram: @ekaklinkova

In [3]:
pip install requests beautifulsoup4 pandas openpyxl lxml

Note: you may need to restart the kernel to use updated packages.


In [4]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from io import BytesIO
import re
import math

In [5]:
# URL страницы
url = 'https://38.rosstat.gov.ru/folder/167937'

try:
    # Получаем содержимое страницы
    response = requests.get(url, timeout=30, verify=False)  # без сертификата
    response.raise_for_status()
    
    # Парсим HTML
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # Ищем ссылку на файл с нужным названием
    target_file = None
    for link in soup.find_all('a', href=True):

        link_text = link.get_text(strip=True)
        href = link['href']
        
        # Проверяем, содержит ли текст ссылки нужное название
        #file_name = 'Численность мужчин и женщин'
        file_name = 'Иркутская область_2012_2021_с учетом итогов Всероссийской переписи населения 2020 года'
        
        if file_name in href:
            # Формируем полный URL
            if href.startswith('http'):
                target_file = href
            elif href.startswith('/'):
                # Если ссылка относительная, добавляем домен
                base_url = 'https://38.rosstat.gov.ru'
                target_file = base_url + href
            else:
                # Если ссылка относительно текущей страницы
                target_file = url.rstrip('/') + '/' + href
            break
    
    if not target_file:
        raise FileNotFoundError("Файл с указанным названием не найден на странице")
    
    print(f"Найден файл: {target_file}")
    
    # Загружаем Excel‑файл
    file_response = requests.get(target_file, timeout=30, verify=False) # без сертификата
    file_response.raise_for_status()
    
    # Создаём объект BytesIO из содержимого файла
    excel_file = BytesIO(file_response.content)
    
    # Читаем Excel в датасет
    # df = pd.read_excel(excel_file)
    
    years = [2016, 2017, 2018, 2019]
    df_population = {}
    
    for year in years:
        page_name = f'на 01.01.{year}'
        # display(page_name)
        df_population[year] = pd.read_excel(excel_file, engine='openpyxl', sheet_name = page_name)
        # df_population[year] = df_population[year].dropna(how='all', axis=1)
        # df_population[year] = df_population[year].dropna(how='all')
        df_population[year] = df_population[year].iloc[7:108]
        df_population[year] = df_population[year].iloc[:,:5]
        df_population[year] =  df_population[year].reset_index(drop=True)
        df_population[year].iloc[100,0] = 100

    
    print("Файл успешно загружен в датасет!")
#    print(f"Размер датасета: {df.shape}")
#    print("\nПервые 5 строк:")
#    print(df.head())
    
except requests.exceptions.RequestException as e:
    print(f"Ошибка при загрузке страницы или файла: {e}")
except FileNotFoundError as e:
    print(e)
except Exception as e:
    print(f"Ошибка при обработке файла: {e}")

C:\Users\Lenovo\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host '38.rosstat.gov.ru'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Найден файл: https://38.rosstat.gov.ru/storage/mediabank/Иркутская область_2012_2021_с учетом итогов Всероссийской переписи населения 2020 года.xlsx


C:\Users\Lenovo\anaconda3\Lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host '38.rosstat.gov.ru'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Файл успешно загружен в датасет!


In [6]:
# year = 2016
df = {}

columns = [
    'Год',
    'Пол',
    'Всего',
    '0-4',
    '5-6',
    '7-14',
    '15-17',
    '18-24',
    '25-34',
    '35-44',
    '45-54',
    '55-64',
    '65+',
    '0-14',
    '15+',
    '0-17'
]

data = []

for year in years:
    for col_number in range(2, df_population[year].shape[1]):
    
        new_row = []    
        new_row.append(year) # Год
        if (col_number == 2): # Пол
            new_row.append('всего')
        if (col_number == 3):
            new_row.append('мужчины')
        if (col_number == 4):
            new_row.append('женщины')

        #Всего
        value = 0
        for row_number in range(0, df_population[year].shape[0]):
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)
    
        # 0-4
        value = 0
        for row_number in range(0, 5):
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)

        # 5-6
        value = 0
        for row_number in range(5, 7): 
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)

        # 7-14
        value = 0
        for row_number in range(7, 15): 
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)
      
        # 15-17
        value = 0
        for row_number in range(15, 18): 
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)

        # 18-24
        value = 0
        for row_number in range(18, 25): 
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)

        # 25-34
        value = 0
        for row_number in range(25, 35): 
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)

        # 35-44
        value = 0
        #for row_number in range(35, 44): 
        for row_number in range(35, 45): 
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)

        # 45-54
        value = 0
        #for row_number in range(45, 54): 
        for row_number in range(45, 55): 
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)

        # 55-64
        value = 0
        #for row_number in range(55, 64): 
        for row_number in range(55, 65):
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)

        # 65+
        value = 0
        for row_number in range(65, df_population[year].shape[0]): 
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)

        # 0-14
        value = 0
        for row_number in range(0, 15): 
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)

        # 15+
        value = 0
        for row_number in range(15, df_population[year].shape[0]): 
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)

        # 0-17
        value = 0
        for row_number in range(0, 18): 
            value += df_population[year].iloc[row_number, col_number]
        new_row.append(value)        

        data.append(new_row)
    
df = pd.DataFrame(data, columns=columns)
    

In [7]:
# Промежуточный результат. Абсолютные значения
df

,Год,Пол,Всего,0-4,5-6,7-14,15-17,18-24,25-34,35-44,45-54,55-64,65+,0-14,15+,0-17
0,2016,всего,2415690,184508,67706,229645,73773,189124,413026,350810,297449,322398,287251,481859,1933831,555632
1,2016,мужчины,1112547,94508,34693,117503,37923,94829,204901,167038,137244,134604,89304,246704,865843,284627
2,2016,женщины,1303143,90000,33013,112142,35850,94295,208125,183772,160205,187794,197947,235155,1067988,271005
3,2017,всего,2412359,183080,69747,237158,74368,176655,405187,354764,290902,322946,297552,489985,1922374,564353
4,2017,мужчины,1110187,94230,35615,121485,38178,88773,201121,168470,134200,135163,92952,251330,858857,289508
5,2017,женщины,1302172,88850,34132,115673,36190,87882,204066,186294,156702,187783,204600,238655,1063517,274845
6,2018,всего,2408221,177110,73839,243151,77090,168455,390490,360708,288189,320906,308283,494100,1914121,571190
7,2018,мужчины,1106926,91120,37669,124728,39169,85109,193483,171197,132990,134602,96859,253517,853409,292686
8,2018,женщины,1301295,85990,36170,118423,37921,83346,197007,189511,155199,186304,211424,240583,1060712,278504
9,2019,всего,2402358,170405,74605,250685,79353,165456,371416,367345,287368,317677,318048,495695,1906663,575048


In [8]:
def round_math(num):
    if num >= 0:
        return math.floor(num + 0.5)
    else:
        return math.ceil(num - 0.5)

In [9]:
df_avg = {}
columns = [
    'Год',
    'Пол',
    'Всего',
    '0-4',
    '5-6',
    '7-14',
    '15-17',
    '18-24',
    '25-34',
    '35-44',
    '45-54',
    '55-64',
    '65+',
    '0-14',
    '15+',
    '0-17'
]
data = []

for row_number in range(0, df.shape[0] - 3):
    new_row = []
    new_row.append(df.iloc[row_number, 0])
    new_row.append(df.iloc[row_number, 1])
    
    for col_number in range(2,df.shape[1]):
        value = round_math((df.iloc[row_number, col_number] + df.iloc[row_number + 3, col_number])/2)
        new_row.append(value)
    data.append(new_row)
df_avg = pd.DataFrame(data, columns=columns)
display(df_avg)

,Год,Пол,Всего,0-4,5-6,7-14,15-17,18-24,25-34,35-44,45-54,55-64,65+,0-14,15+,0-17
0,2016,всего,2414025,183794,68727,233402,74071,182890,409107,352787,294176,322672,292402,485922,1928103,559993
1,2016,мужчины,1111367,94369,35154,119494,38051,91801,203011,167754,135722,134884,91128,249017,862350,287068
2,2016,женщины,1302658,89425,33573,113908,36020,91089,206096,185033,158454,187789,201274,236905,1065753,272925
3,2017,всего,2410290,180095,71793,240155,75729,172555,397839,357736,289546,321926,302918,492043,1918248,567772
4,2017,мужчины,1108557,92675,36642,123107,38674,86941,197302,169834,133595,134883,94906,252424,856133,291097
5,2017,женщины,1301734,87420,35151,117048,37056,85614,200537,187903,155951,187044,208012,239619,1062115,276675
6,2018,всего,2405290,173758,74222,246918,78222,166956,380953,364027,287779,319292,313166,494898,1910392,573119
7,2018,мужчины,1105027,89448,37978,126670,39603,84422,188693,172916,132626,134038,98635,254095,850932,293697
8,2018,женщины,1300263,84310,36245,120249,38619,82534,192261,191111,155153,185254,214531,240803,1059460,279422


In [10]:
path = 'E:/IrkutskProject/Data/'
df_avg.to_csv(f'{path}Common_2016-2018/avg_population_2016-2018.csv', index=False)